# Colab Experiment Runner

Run this notebook with a Colab GPU runtime connected (via PyCharm or directly at colab.research.google.com).

**Start here: run the Setup cell.** It tells you whether your local files are visible to the kernel.
If they are, everything just works — magic GPU mode. If not, flip `USE_GIT = True` and it handles the rest.

## Setup

Fill in your HuggingFace token below, then run this cell. It auto-detects whether your local files are accessible.

**One-time steps on the HuggingFace website (any device, any browser):**
- Accept LLaMA license → huggingface.co/meta-llama/Llama-3.1-8B-Instruct
- Accept Gemma 2 license → huggingface.co/google/gemma-2-9b-it
- Get your token → huggingface.co → Settings → Access Tokens

In [ ]:
import os, sys

# ── Only thing you need to fill in ────────────────────────────────────────────
HF_TOKEN = ''          # paste your HuggingFace token here (starts with hf_...)
USE_GIT  = False       # flip to True ONLY if the file-check below prints ✗
GITHUB_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'   # only needed if USE_GIT=True
BRANCH     = 'Ammarsbranch'                                      # only needed if USE_GIT=True
# ──────────────────────────────────────────────────────────────────────────────

REPO_DIR = 'llm-invisible-watermarking'

if USE_GIT:
    # ── Git mode: pull repo from GitHub into Colab's filesystem ──────────────
    if not os.path.exists(REPO_DIR):
        print(f'Cloning {BRANCH}...')
        os.system(f'git clone --branch {BRANCH} {GITHUB_URL} {REPO_DIR}')
    else:
        print('Pulling latest...')
        os.system(f'cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull')
    os.chdir(REPO_DIR)
else:
    # ── Magic GPU mode: assume PyCharm synced local files to this kernel ──────
    # Try to find the repo — check cwd and common Colab paths
    candidates = [
        '.',
        f'/content/{REPO_DIR}',
        os.path.join(os.path.expanduser('~'), REPO_DIR),
    ]
    for path in candidates:
        if os.path.exists(os.path.join(path, 'watermark')):
            os.chdir(path)
            print(f'Found repo at: {os.path.abspath(path)}')
            break
    else:
        print(f'Current dir: {os.getcwd()}')
        print('Repo not found at expected paths. If the check below fails, set USE_GIT = True.')

sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')

# ── Check: can we see the local source files? ─────────────────────────────────
print()
try:
    from watermark.logits_processor import WatermarkLogitsProcessor
    from watermark.detector import WatermarkDetector
    from pipeline.generate import CorpusGenerator, load_corpus
    print('✓ Source files are accessible to this kernel')
    print('  (magic GPU mode is working — no git needed)')
except ImportError as e:
    print(f'✗ Source files NOT accessible: {e}')
    print('  → Set USE_GIT = True above and re-run this cell')

# ── Install packages (always needed — Colab resets packages each session) ─────
print()
print('Installing packages...')
os.system('pip install -q transformers datasets accelerate scipy numpy matplotlib tqdm sentencepiece protobuf')
print('✓ Packages installed')

# ── HuggingFace login (needed to download LLaMA and Gemma 2 weights) ─────────
print()
if HF_TOKEN:
    os.system(f'huggingface-cli login --token {HF_TOKEN}')
    print('✓ Logged into HuggingFace')
else:
    print('HF_TOKEN is empty — paste your token above, or run: !huggingface-cli login')

# ── GPU check ─────────────────────────────────────────────────────────────────
print()
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU: {name}  ({vram:.1f} GB VRAM)')
    if vram < 14:
        print('  WARNING: <14 GB VRAM — LLaMA 8B fp16 needs ~14 GB. Use A100 runtime.')
else:
    print('✗ No GPU — switch to a GPU runtime in Colab (Runtime → Change runtime type)')

---
## Step 1 — Verify watermark algorithm
**~5 seconds. No model download.**

Checks that the processor and detector produce identical green lists from the same hash.
This is the most important correctness check — if it fails, detection won't work.

In [ ]:
import sys, random, torch
sys.path.insert(0, '.')

from watermark.logits_processor import WatermarkLogitsProcessor
from watermark.detector import WatermarkDetector

proc = WatermarkLogitsProcessor(vocab_size=32000, gamma=0.5, seed=42)
det  = WatermarkDetector(vocab_size=32000, gamma=0.5, seed=42)

# Green lists must match for every token
for tok in [0, 1, 1234, 31999]:
    p = set(proc._get_greenlist_ids(tok).tolist())
    d = det._get_greenlist_set(tok)
    assert p == d, f'HASH MISMATCH at token {tok}'
print('✓ Processor and detector produce identical green lists')

# Delta applied to green logits only
scores = torch.zeros(1, 32000)
mod    = proc(torch.tensor([[1234]]), scores.clone())
green  = proc._get_greenlist_ids(1234).tolist()
assert all(mod[0, g].item() > 0 for g in green[:10])
assert mod[0, 0].item() == 0.0
print('✓ Delta applied to green logits only')

# Forced-green sequence → high z-score
random.seed(999)
toks = [random.randint(0, 31999)]
for _ in range(200):
    toks.append(random.choice(proc._get_greenlist_ids(toks[-1]).tolist()))
r = det.score_sequence(toks)
print(f'✓ Forced-green sequence: z={r.z_score:.2f}  green={r.green_fraction:.0%}')
assert r.z_score > 10

# Random sequence → not detected
null = [random.randint(0, 31999) for _ in range(200)]
nr   = det.score_sequence(null)
print(f'✓ Random sequence:       z={nr.z_score:.2f}  green={nr.green_fraction:.0%}')
assert abs(nr.z_score) < 4

print('\n✅  Step 1 PASSED — algorithm is correct')

---
## Step 2 — Generate LLaMA corpus
**~2-4 hours. Fully resumable — if Colab disconnects, just re-run Setup then re-run this cell.**

150 prompts × 3 datasets × 2 (watermarked + control) = 900 completions.  
Output: `results/corpus_llama_d2.jsonl`

In [ ]:
!python scripts/generate_corpus.py

In [ ]:
# Sanity check — run after corpus generation finishes
from pipeline.generate import load_corpus
import numpy as np

corpus = load_corpus('results/corpus_llama_d2.jsonl')
wm  = [x for x in corpus if x['watermarked']]
uwm = [x for x in corpus if not x['watermarked']]
print(f'Watermarked:   {len(wm)}')
print(f'Unwatermarked: {len(uwm)}')
print(f'Avg tokens  — wm: {np.mean([x["n_tokens"] for x in wm]):.0f},  '
      f'uwm: {np.mean([x["n_tokens"] for x in uwm]):.0f}')
print(f'Sources: {set(x["source"] for x in corpus)}')
assert len(wm) > 200
print('\n✅  Corpus looks good')

---
## Step 3 — Detection + perplexity eval
**~30 min.**

Output: `results/detection_llama_summary.json` + `results/detection_llama_zscores.npz`  
Acceptance criterion: TPR @ 1% FPR on ≥150-token completions > 0.90

In [ ]:
!python scripts/eval_detection.py

In [ ]:
import json
with open('results/detection_llama_summary.json') as f:
    s = json.load(f)
print(f"Calibrated threshold:   {s['calibrated_z_threshold']:.3f}")
print(f"TPR (all lengths):      {s['tpr_at_1pct_fpr_all']:.1%}")
long_key = next((k for k in s if 'ge150' in k), None)
if long_key:
    v = s[long_key]
    status = '✅' if v > 0.90 else '❌'
    print(f"TPR (>=150 tokens):     {v:.1%}  {status}  (need >0.90)")
print(f"PPL ratio (wm/uwm):     {s['ppl_ratio_wm_over_uwm']:.3f}  (ideally close to 1.0)")

---
## Step 4 — Length curves
**~10 min. No model needed.**

Output: `results/length_curves_llama.json`

In [ ]:
!python scripts/eval_length.py

In [ ]:
import json
with open('results/length_curves_llama.json') as f:
    lc = json.load(f)
print(f"{'tokens':>8}  {'TPR':>6}")
for row in lc:
    bar = '█' * int(row['tpr'] * 25)
    print(f"{row['n_tokens']:>8}  {row['tpr']:>6.3f}  {bar}")

---
## Step 5 — Robustness sweep
**~15 min basic / ~90 min with LLM paraphrase.**

Output: `results/robustness_llama.json`

In [ ]:
# Basic: word substitution + token insert/delete (~15 min)
!python scripts/eval_robustness.py

# Uncomment to also run LLM paraphrase attack (~90 min, loads LLaMA again):
# !python scripts/eval_robustness.py --with-paraphrase

In [ ]:
import json
with open('results/robustness_llama.json') as f:
    rob = json.load(f)
print(f"{'Condition':<32} {'TPR':>6}")
for cond, vals in rob.items():
    bar = '█' * int(vals['tpr_at_1pct_fpr'] * 25)
    print(f"{cond:<32} {vals['tpr_at_1pct_fpr']:>6.1%}  {bar}")

---
## Step 6 — Delta sweep
**~2-3 hours.**

Output: `results/delta_sweep_llama.json`

In [ ]:
!python scripts/eval_delta_sweep.py

In [ ]:
import json
with open('results/delta_sweep_llama.json') as f:
    sweep = json.load(f)
print(f"{'delta':>7} {'TPR':>7} {'PPL_wm':>8} {'ratio':>7}")
for row in sweep:
    print(f"{row['delta']:>7.1f} {row['tpr']:>7.3f} {row['mean_ppl_wm']:>8.1f} {row['ppl_ratio']:>7.3f}")

---
## Step 7 — Gemma 2 9B (secondary model)
**~1-2 hours generation + ~20 min eval.**

Make sure you accepted the Gemma 2 license at huggingface.co/google/gemma-2-9b-it first.

In [ ]:
!python scripts/generate_corpus.py --model google/gemma-2-9b-it --n-per-dataset 67

In [ ]:
!python scripts/eval_detection.py --model google/gemma-2-9b-it

In [ ]:
import json
print(f"{'Model':<20} {'TPR(all)':>10} {'TPR(150t)':>10} {'PPL_ratio':>10}")
print('-' * 55)
for fname, label in [('results/detection_llama_summary.json', 'LLaMA 3.1 8B'),
                     ('results/detection_gemma_summary.json', 'Gemma 2 9B')]:
    try:
        with open(fname) as f:
            s = json.load(f)
        long_key = next((k for k in s if 'ge150' in k), None)
        tpr_long = f"{s[long_key]:.1%}" if long_key else 'n/a'
        print(f"{label:<20} {s['tpr_at_1pct_fpr_all']:>10.1%} {tpr_long:>10} {s['ppl_ratio_wm_over_uwm']:>10.3f}")
    except FileNotFoundError:
        print(f"{label:<20} not yet run")

---
## Step 8 — Generate figures
**~2 min.**

Output: `figures/fig{2-6}.pdf`  
Fig 1 is a placeholder (drawn manually in TikZ) — `figures/fig1_method.txt` has the description.

In [ ]:
!python scripts/make_figures.py

In [ ]:
from pathlib import Path
print('Generated figures:')
for f in sorted(Path('figures').glob('*.pdf')):
    print(f'  {f.name}  ({f.stat().st_size // 1024} KB)')

---
## Download results

Downloads a zip of `results/` + `figures/` to your local machine.  
**Run before the Colab session ends** — Colab deletes everything in `/content/` when the session closes.

(Skip this cell entirely if magic GPU mode is working and files are already landing locally.)

In [ ]:
import zipfile
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
archive   = f'/content/watermark_results_{timestamp}.zip'

with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['results', 'figures']:
        for fp in Path(folder).rglob('*'):
            if fp.is_file():
                zf.write(fp)

size_mb = Path(archive).stat().st_size / 1e6
print(f'Created {archive}  ({size_mb:.1f} MB)')

from google.colab import files
files.download(archive)